# 05 - Ensambles de Pronósticos

**Módulo 3 - Series de Tiempo | ML Avanzado**

Combinar pronósticos es una de las ideas más antiguas y robustas del área
(Bates & Granger, 1969). La evidencia empírica es contundente: en las
competencias **M3/M4**, las combinaciones simples vencieron sistemáticamente a
casi todos los modelos individuales.

**¿Por qué funciona?** Cada modelo comete errores distintos: SARIMA captura la
autocorrelación lineal, Holt-Winters el nivel/estacionalidad suave, XGBoost
las no-linealidades del calendario. Si los errores no están perfectamente
correlacionados, promediar **cancela parte del error** — el mismo argumento
varianza-reducción del bagging (Módulo 2), aplicado a pronósticos.

La paradoja conocida como *forecast combination puzzle*: la **media simple**
es dificilísima de vencer con esquemas de pesos "óptimos", porque los pesos
estimados añaden su propia varianza.

Probaremos cuatro combinaciones:

1. **Media simple** — $\hat y = \frac{1}{M}\sum_m \hat y^{(m)}$
2. **Mediana** — robusta a un modelo que se descarrile.
3. **Pesos por inverso del error** — $w_m \propto 1 / \text{RMSE}^{(m)}_{val}$
4. **Stacking** — un meta-modelo lineal aprende los pesos sobre una ventana de
   validación (el análogo temporal del stacking del Módulo 2).

In [ ]:
import os, sys, warnings
warnings.filterwarnings("ignore")

# Hacemos importable utils/ tanto si el notebook corre desde notebooks/ como
# desde la raíz del repositorio.
_here = os.getcwd()
for cand in (os.path.join(_here, "..", "utils"), os.path.join(_here, "utils"),
             os.path.join(_here, "..", "..", "module3-time-series", "utils")):
    cand = os.path.abspath(cand)
    if os.path.isdir(cand) and cand not in sys.path:
        sys.path.insert(0, cand)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
from mlflow_helpers import setup_mlflow, log_and_register, register_best_run

plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["axes.grid"] = True
np.random.seed(42)
print("Versión de MLflow:", mlflow.__version__)

In [ ]:
# ---------------------------------------------------------------------------
# Carga del dataset UCI #235 (con caché local y respaldo sintético offline)
# ---------------------------------------------------------------------------
import io, zipfile, urllib.request

UCI_ZIP_URL = ("https://archive.ics.uci.edu/static/public/235/"
               "individual+household+electric+power+consumption.zip")

def _data_dir():
    for cand in ("../data", "data", "module3-time-series/data"):
        cand = os.path.abspath(cand)
        if os.path.isdir(cand):
            return cand
    cand = os.path.abspath("../data")
    os.makedirs(cand, exist_ok=True)
    return cand

DATA_DIR = _data_dir()
DAILY_CSV = os.path.join(DATA_DIR, "household_power_daily.csv")
HOURLY_CSV = os.path.join(DATA_DIR, "household_power_hourly.csv")

def load_household_power():
    """Devuelve (daily, hourly): potencia activa global media, en kW."""
    if os.path.isfile(DAILY_CSV) and os.path.isfile(HOURLY_CSV):
        daily = pd.read_csv(DAILY_CSV, index_col=0, parse_dates=True).iloc[:, 0]
        hourly = pd.read_csv(HOURLY_CSV, index_col=0, parse_dates=True).iloc[:, 0]
        return daily.asfreq("D"), hourly.asfreq("h")

    print("Descargando el dataset UCI #235 (~20 MB)...")
    raw = urllib.request.urlopen(UCI_ZIP_URL, timeout=180).read()
    with zipfile.ZipFile(io.BytesIO(raw)) as zf:
        with zf.open("household_power_consumption.txt") as fh:
            df = pd.read_csv(fh, sep=";", na_values=["?"], low_memory=False,
                             usecols=["Date", "Time", "Global_active_power"])
    ts = pd.to_datetime(df["Date"] + " " + df["Time"],
                        format="%d/%m/%Y %H:%M:%S")
    power = pd.Series(df["Global_active_power"].astype(float).to_numpy(),
                      index=ts, name="global_active_power_kw").sort_index()

    # Agregamos y rellenamos huecos por interpolación temporal (~1.25% de
    # minutos faltantes + un par de cortes de varios días).
    daily = power.resample("D").mean().interpolate(method="time")
    hourly = power.resample("h").mean().interpolate(method="time")
    daily = daily.iloc[1:-1]                       # primer/último día parciales
    hourly = hourly.loc[daily.index.min():
                        daily.index.max() + pd.Timedelta(hours=23)]
    daily.to_frame().to_csv(DAILY_CSV)
    hourly.to_frame().to_csv(HOURLY_CSV)
    return daily.asfreq("D"), hourly.asfreq("h")

try:
    daily, hourly = load_household_power()
    print(f"daily : {daily.index.min().date()} .. {daily.index.max().date()} "
          f"(n={len(daily)})")
    print(f"hourly: n={len(hourly)}")
except Exception as e:
    print("No se pudo descargar el dataset:", repr(e))
    print("Usando RESPALDO SINTÉTICO (estacionalidad semanal + anual).")
    rng = np.random.default_rng(7)
    idx = pd.date_range("2006-12-17", "2010-11-25", freq="D")
    t = np.arange(len(idx))
    daily = pd.Series(
        1.1
        + 0.35 * np.cos(2 * np.pi * (t - 20) / 365.25)   # invierno alto
        + 0.10 * (idx.dayofweek >= 5)                     # fin de semana
        + rng.normal(0, 0.12, len(idx)),
        index=idx, name="global_active_power_kw").clip(lower=0.1).asfreq("D")
    hidx = pd.date_range(idx.min(), idx.max() + pd.Timedelta(hours=23), freq="h")
    hh = hidx.hour.to_numpy()
    base = daily.reindex(pd.DatetimeIndex(hidx.date)).to_numpy()
    profile = 0.6 + 0.35 * np.sin(2 * np.pi * (hh - 14) / 24) \
              + 0.25 * ((hh >= 18) & (hh <= 22))
    hourly = pd.Series(base * profile + rng.normal(0, 0.05, len(hidx)),
                       index=hidx, name=daily.name).clip(lower=0.05).asfreq("h")

In [ ]:
# ---------------------------------------------------------------------------
# Métricas de pronóstico + gráfico estándar — se usan en TODOS los notebooks.
# ---------------------------------------------------------------------------
def forecast_metrics(y_true, y_pred):
    """MSE, RMSE, MAE, MAPE y sMAPE como dict {nombre: float}."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    err = y_true - y_pred
    mse = float(np.mean(err ** 2))
    return {
        "MSE":   mse,
        "RMSE":  float(np.sqrt(mse)),
        "MAE":   float(np.mean(np.abs(err))),
        "MAPE":  float(np.mean(np.abs(err) / np.abs(y_true)) * 100.0),
        "sMAPE": float(np.mean(2.0 * np.abs(err)
                               / (np.abs(y_true) + np.abs(y_pred))) * 100.0),
    }

def print_metrics(name, m):
    print(f"{name:<26s} MSE={m['MSE']:.4f}  RMSE={m['RMSE']:.4f}  "
          f"MAE={m['MAE']:.4f}  MAPE={m['MAPE']:.2f}%  sMAPE={m['sMAPE']:.2f}%")

def metrics_table(metrics_by_model):
    """dict {modelo: dict_de_métricas} -> DataFrame ordenado por sMAPE."""
    return (pd.DataFrame(metrics_by_model).T
            .sort_values("sMAPE").round(4))

def plot_forecast(train, test, forecasts, title="", tail=180, ci=None):
    """Cola del train + test real + uno o varios pronósticos.

    forecasts : dict {nombre: pd.Series indexada como test}
    ci        : tupla opcional (lower, upper) para una banda de confianza
    Devuelve la figura (útil para loggearla en MLflow).
    """
    fig, ax = plt.subplots(figsize=(13, 5))
    train.iloc[-tail:].plot(ax=ax, label="train (cola)", color="0.65")
    test.plot(ax=ax, label="real (test)", color="black", lw=2)
    for name, fc in forecasts.items():
        fc.plot(ax=ax, label=name, lw=1.8)
    if ci is not None:
        ax.fill_between(test.index, ci[0], ci[1], alpha=0.2, label="IC 95%")
    ax.set_title(title)
    ax.set_ylabel("potencia activa media (kW)")
    ax.legend()
    plt.tight_layout()
    plt.show()
    return fig

In [ ]:
# ---------------------------------------------------------------------------
# Matriz de variables supervisada para pronóstico diario (sin fuga temporal).
# Construida y explicada en el notebook 03 — aquí la reutilizamos tal cual.
# ---------------------------------------------------------------------------
def make_features(s, lags=(1, 2, 3, 7, 14, 28, 365),
                  roll_windows=(7, 28), fourier=((7, 2), (365.25, 3))):
    df = pd.DataFrame({"y": s})
    df["t_index"] = np.arange(len(df))

    # rezagos
    for L in lags:
        df[f"lag_{L}"] = df["y"].shift(L)

    # estadísticos móviles SOLO del pasado: shift(1) antes de rolling
    past = df["y"].shift(1)
    for w in roll_windows:
        df[f"rollmean_{w}"] = past.rolling(w).mean()
        df[f"rollstd_{w}"] = past.rolling(w).std()
        df[f"rollmin_{w}"] = past.rolling(w).min()
        df[f"rollmax_{w}"] = past.rolling(w).max()

    # calendario
    df["dayofweek"] = df.index.dayofweek
    df["month"] = df.index.month
    df["is_weekend"] = (df.index.dayofweek >= 5).astype(int)

    # estacionalidad de Fourier (semanal y anual)
    tt = df["t_index"].to_numpy()
    for period, K in fourier:
        for k in range(1, K + 1):
            df[f"sin_{int(period)}_{k}"] = np.sin(2 * np.pi * k * tt / period)
            df[f"cos_{int(period)}_{k}"] = np.cos(2 * np.pi * k * tt / period)
    return df

## 1. Tres ventanas: train / validación / test

Para aprender pesos sin hacer trampa necesitamos una ventana de **validación**
separada del test final:

```
|--------------- train ---------------|-- val (60d) --|-- test (60d) --|
```

- Los modelos base se ajustan en *train* y pronostican *val* → con esos
  errores estimamos los **pesos**.
- Luego se reajustan en *train+val* y pronostican *test* → ahí comparamos
  **todo** (bases y ensambles) de forma honesta.

In [ ]:
H = 60
trainval, test = daily.iloc[:-H], daily.iloc[-H:]
train, val = trainval.iloc[:-H], trainval.iloc[-H:]
print(f"train: n={len(train)} | val: n={len(val)} | test: n={len(test)}")

## 2. Los modelos base

Cuatro pronosticadores de familias distintas (la **diversidad** es el
ingrediente clave de un ensamble): naive estacional, Holt-Winters, SARIMA y
XGBoost recursivo — los mismos de los notebooks 01, 02 y 04.

In [ ]:
def xgb_recursive_forecast(model, history, h):
    """Pronóstico multi-paso RECURSIVO: extiende la serie un día a la vez,
    recalculando las variables (los rezagos recientes van siendo predichos)."""
    hist = history.copy()
    preds = []
    for _ in range(h):
        next_date = hist.index[-1] + pd.Timedelta(days=1)
        f_next = make_features(
            pd.concat([hist, pd.Series([np.nan], index=[next_date])]))
        row = f_next.drop(columns=["y"]).iloc[[-1]]
        yhat = float(model.predict(row)[0])
        hist = pd.concat([hist, pd.Series([yhat], index=[next_date])])
        preds.append(yhat)
    idx = pd.date_range(history.index[-1] + pd.Timedelta(days=1),
                        periods=h, freq="D")
    return pd.Series(preds, index=idx, name="xgb_recursive")

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX
from xgboost import XGBRegressor

XGB_PARAMS = dict(n_estimators=500, learning_rate=0.05, max_depth=5,
                  subsample=0.8, colsample_bytree=0.8,
                  random_state=42, n_jobs=-1)

def fc_snaive(history, h, idx):
    vals = np.tile(history.iloc[-7:].to_numpy(), h // 7 + 1)[:h]
    return pd.Series(vals, index=idx)

def fc_holtwinters(history, h, idx):
    m = ExponentialSmoothing(history, trend="add", damped_trend=True,
                             seasonal="add", seasonal_periods=7,
                             initialization_method="estimated").fit()
    f = m.forecast(h); f.index = idx
    return f

def fc_sarima(history, h, idx):
    m = SARIMAX(history, order=(1, 0, 1), seasonal_order=(0, 1, 1, 7),
                enforce_stationarity=False, enforce_invertibility=False
                ).fit(disp=False)
    f = m.forecast(h); f.index = idx
    return f

def fc_xgboost(history, h, idx):
    feat = make_features(history).dropna()
    model = XGBRegressor(**XGB_PARAMS).fit(feat.drop(columns=["y"]), feat["y"])
    f = xgb_recursive_forecast(model, history, h); f.index = idx
    return f, model

BASE = {"snaive": fc_snaive, "holt_winters": fc_holtwinters,
        "sarima": fc_sarima, "xgboost": fc_xgboost}

In [ ]:
# Pronósticos sobre VALIDACIÓN (ajuste en train)...
F_val = {}
for name, fn in BASE.items():
    out = fn(train, H, val.index)
    F_val[name] = out[0] if isinstance(out, tuple) else out
F_val = pd.DataFrame(F_val)

# ...y sobre TEST (reajuste en train+val)
F_test, stack_xgb_model = {}, None
for name, fn in BASE.items():
    out = fn(trainval, H, test.index)
    if isinstance(out, tuple):
        F_test[name], stack_xgb_model = out
    else:
        F_test[name] = out
F_test = pd.DataFrame(F_test)

print("errores de VALIDACIÓN (sirven para calcular pesos):")
val_rmse = {}
for name in F_val:
    m = forecast_metrics(val, F_val[name])
    val_rmse[name] = m["RMSE"]
    print_metrics(name, m)

## 3. Las combinaciones

**Pesos por inverso del RMSE de validación** (normalizados a que sumen 1):

$$
w_m = \frac{1 / \text{RMSE}_m}{\sum_j 1 / \text{RMSE}_j}
\qquad
\hat y^{comb}_t = \sum_m w_m\, \hat y^{(m)}_t .
$$

**Stacking**: regresión lineal **sin intercepto y con pesos no negativos**
(para que sea una combinación convexa interpretable) de $y_{val}$ sobre la
matriz de pronósticos de validación.

In [ ]:
from sklearn.linear_model import LinearRegression

ens = {}
ens["media_simple"] = F_test.mean(axis=1)
ens["mediana"] = F_test.median(axis=1)

w = pd.Series({m: 1.0 / r for m, r in val_rmse.items()})
w /= w.sum()
ens["pesos_inv_rmse"] = (F_test * w).sum(axis=1)
print("pesos inverso-RMSE:", w.round(3).to_dict())

stacker = LinearRegression(positive=True, fit_intercept=False)
stacker.fit(F_val, val)
coef = pd.Series(stacker.coef_, index=F_val.columns)
ens["stacking"] = pd.Series(stacker.predict(F_test), index=test.index)
print("coeficientes stacking:", coef.round(3).to_dict())

# Nota: el stacking puede concentrar todo el peso en 1-2 modelos; los pesos
# inverso-RMSE reparten de forma más conservadora. Compararlos es el punto.

## 4. Evaluación final sobre el test

In [ ]:
all_metrics = {}
for name in F_test:
    all_metrics[f"base: {name}"] = forecast_metrics(test, F_test[name])
for name, fc in ens.items():
    all_metrics[f"ens: {name}"] = forecast_metrics(test, fc)

table = metrics_table(all_metrics)
display(table)

colors = ["C1" if i.startswith("ens") else "C0" for i in table.index]
ax = table["sMAPE"].plot(kind="barh", figsize=(9, 4.5), color=colors)
ax.set_xlabel("sMAPE (%)  (menor = mejor)")
ax.set_title("Modelos base (azul) vs ensambles (naranja) - test 60 días")
plt.tight_layout(); plt.show()

### Lectura de los resultados

En esta serie **XGBoost domina** con claridad a las demás familias. Cuando eso
pasa, las combinaciones de peso uniforme (media, mediana) **diluyen** al mejor
modelo, y el ensamble más competitivo es el **stacking**, que aprendió en
validación a concentrar el peso en el dominante (míralo en sus coeficientes).

La lección del *forecast combination puzzle* aplica cuando los modelos base
son **comparablemente buenos y diversos** — ahí la media simple brilla. La
moraleja práctica es otra: el ensamble es un **seguro contra apostarle al
modelo equivocado** (ex ante no sabes cuál base ganará en producción), no una
garantía de mejora sobre el mejor modelo visto en retrospectiva.

In [ ]:
fig_ens = plot_forecast(
    trainval, test,
    {"mejor base": F_test[min(val_rmse, key=val_rmse.get)],
     "media_simple": ens["media_simple"],
     "stacking": ens["stacking"]},
    title="Ensambles de pronósticos vs el mejor modelo base")

## 5. MLflow: un run por método + registry

Registramos bases y ensambles en el experimento `module3-05-ensembles`. El
meta-modelo de stacking (el único ensamble con un objeto entrenado) se publica
en el registry como `module3-power-stacking`.

In [ ]:
setup_mlflow("module3-05-ensembles", backend="dagshub")

for name in F_test:
    log_and_register(
        run_name=f"base-{name}",
        params={"tipo": "base", "modelo": name, "horizon_days": H,
                "dataset": "uci-household-power"},
        metrics=all_metrics[f"base: {name}"],
        tags={"notebook": "05_ensembles", "familia": "base"},
    )

ens_params = {
    "media_simple":   {"combinacion": "mean"},
    "mediana":        {"combinacion": "median"},
    "pesos_inv_rmse": {"combinacion": "inv_rmse",
                       **{f"w_{k}": round(float(v), 4) for k, v in w.items()}},
    "stacking":       {"combinacion": "stacking_lr_positive",
                       **{f"coef_{k}": round(float(v), 4)
                          for k, v in coef.items()}},
}
for name in ens:
    log_and_register(
        run_name=f"ensemble-{name}",
        params={"tipo": "ensemble", **ens_params[name], "horizon_days": H,
                "dataset": "uci-household-power"},
        metrics=all_metrics[f"ens: {name}"],
        model=stacker if name == "stacking" else None,
        flavor="sklearn",
        registered_model_name=("module3-power-stacking"
                               if name == "stacking" else None),
        input_example=F_val.head(3) if name == "stacking" else None,
        tags={"notebook": "05_ensembles", "familia": "ensemble"},
        figures={"plots/comparacion.png": fig_ens},
    )

## 6. Serving: consumir el stacker desde el Registry

Cerramos el **ciclo de gestión del modelo** consumiendo el meta-modelo desde
el registry con el wrapper genérico **`pyfunc`** (`.predict(DataFrame)`, el
mismo contrato que expone `mlflow models serve`).

Servir un ensamble deja una lección extra: el meta-modelo **no basta**. Sus
*features* son los pronósticos de los modelos base, así que el pipeline de
serving completo es

1. reajustar/pronosticar los **modelos base** sobre los datos frescos
   (aquí ese rol lo cumple `F_test`), y
2. pasar esa matriz al **stacker** cargado del registry.

Si los base cambian (versión, orden de columnas), el meta-modelo servido se
rompe en silencio — por eso la **firma** registrada valida columnas y tipos
en cada `predict`.

In [ ]:
MODEL_NAME = "module3-power-stacking"
MODEL_URI = f"models:/{MODEL_NAME}/latest"

serving_stacker = mlflow.pyfunc.load_model(MODEL_URI)
print("Firma del modelo (las columnas son los pronósticos base):")
print(serving_stacker.metadata.signature)

# Paso 1 del pipeline de serving: pronósticos base (ya calculados en F_test).
# Paso 2: el meta-modelo del registry los combina.
fc_serving = pd.Series(
    np.asarray(serving_stacker.predict(F_test)).ravel(), index=test.index)

print_metrics("stacking servido (registry)",
              forecast_metrics(test, fc_serving))
print("¿Idéntico al stacker en memoria?",
      bool(np.allclose(fc_serving.to_numpy(), ens["stacking"].to_numpy())))

fig, ax = plt.subplots(figsize=(12, 4))
test.plot(ax=ax, color="black", lw=2, label="realidad (test)")
fc_serving.plot(ax=ax, color="C3", lw=2,
                label="stacking servido desde el registry")
ax.set_ylabel("kW"); ax.legend()
ax.set_title(f"Serving: {MODEL_NAME}/latest combinando los pronósticos base")
plt.tight_layout(); plt.show()

## Resumen

- **Combinar pronósticos** de familias diversas cancela errores no
  correlacionados — el hallazgo más repetido de las competencias M3/M4.
- La **media simple** es un punto de partida durísimo de vencer (*forecast
  combination puzzle*): los pesos estimados añaden varianza.
- Alternativas: **mediana** (robusta), **pesos por inverso del RMSE de
  validación** (conservadora) y **stacking** con regresión no negativa
  (agresiva; necesita ventana de validación honesta).
- Si un modelo base **domina** (aquí XGBoost), los pesos uniformes lo diluyen
  y el stacking es el que mejor lo recupera. El ensamble es un *seguro*, no
  una garantía: repórtalo siempre junto a los modelos base.
- Extensiones que valen la pena: modelos **híbridos** (SARIMA + ML sobre sus
  residuos) y re-estimar pesos con ventana rodante.
- Todo quedó versionado en MLflow; el stacker vive en el **registry** y lo
  consumimos de vuelta (`models:/module3-power-stacking/latest`) — pero servir
  un ensamble exige servir también su pipeline de pronósticos base.

Siguiente: **06 — Clustering de perfiles de carga con DTW**.